# Google AlloyDB for PostgreSQL - `AlloyDBModelManager`

> [AlloyDB](https://cloud.google.com/alloydb) is a fully managed relational database service that offers high performance, seamless integration, and impressive scalability. AlloyDB is 100% compatible with PostgreSQL. Extend your database application to build AI-powered experiences leveraging AlloyDB's LlamaIndex integrations.

This notebook goes over how to register an [AlloyDB AI model endpoint](https://cloud.google.com/alloydb/docs/ai/register-model-endpoint) with the `AlloyDBModelManager` class, and how to use it in the database.

Learn more about the package on [GitHub](https://github.com/googleapis/llama-index-alloydb-pg-python/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googleapis/llama-index-alloydb-pg-python/blob/main/samples/llama_index_model_manager.ipynb)

## Before you begin

To run this notebook, you will need to do the following:

 * [Create a Google Cloud Project](https://developers.google.com/workspace/guides/create-project)
 * [Enable the AlloyDB API](https://console.cloud.google.com/flows/enableapi?apiid=alloydb.googleapis.com)
 * [Create a AlloyDB cluster and instance.](https://cloud.google.com/alloydb/docs/cluster-create)
 * [Create a AlloyDB database.](https://cloud.google.com/alloydb/docs/quickstart/create-and-connect)
 * [Add a User to the database.](https://cloud.google.com/alloydb/docs/database-users/about)
 * [Enable model endpoint management](https://cloud.google.com/alloydb/docs/ai/register-model-endpoint): install the `google_ml_integration` extension and set the `google_ml_integration.enable_model_support` database flag to `on`.

### 🦙 Library Installation
Install the integration library, `llama-index-alloydb-pg`.

In [ ]:
%pip install --upgrade --quiet llama-index-alloydb-pg

### 🔐 Authentication
Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

* If you are using Colab to run this notebook, use the cell below and continue.
* If you are using Vertex AI Workbench, check out the setup instructions [here](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env).

In [ ]:
from google.colab import auth

auth.authenticate_user()

### Set AlloyDB database values
Find your database values, in the [AlloyDB Instances page](https://console.cloud.google.com/alloydb/clusters).

In [ ]:
# @title Set Your Values Here { display-mode: "form" }
PROJECT_ID = "my-project-id"  # @param {type: "string"}
REGION = "us-central1"  # @param {type: "string"}
CLUSTER = "my-cluster"  # @param {type: "string"}
INSTANCE = "my-primary"  # @param {type: "string"}
DATABASE = "my-database"  # @param {type: "string"}
USER = "postgres"  # @param {type: "string"}
PASSWORD = "my-password"  # @param {type: "string"}

### Create an `AlloyDBEngine`

The `AlloyDBEngine` configures a connection pool to your AlloyDB database.

In [ ]:
from llama_index_alloydb_pg import AlloyDBEngine

engine = await AlloyDBEngine.afrom_instance(
    project_id=PROJECT_ID,
    region=REGION,
    cluster=CLUSTER,
    instance=INSTANCE,
    database=DATABASE,
    user=USER,
    password=PASSWORD,
)

## Create an `AlloyDBModelManager`

`AlloyDBModelManager.create` checks that the `google_ml_integration` extension is installed and that model endpoint management is enabled, and raises a `ValueError` that says what to fix otherwise.

In [ ]:
from llama_index_alloydb_pg import AlloyDBModelManager

manager = await AlloyDBModelManager.create(engine)

## Register a model endpoint

Register the Vertex AI `text-embedding-005` model under your own model ID. For the `google` provider, set `model_qualified_name` (or `model_request_url`).

In [ ]:
MODEL_ID = "my-text-embedding"

await manager.acreate_model(
    MODEL_ID,
    model_provider="google",
    model_type="text_embedding",
    model_qualified_name="text-embedding-005",
)

print(await manager.aget_model(MODEL_ID))
print([model.model_id for model in await manager.alist_models()])

## Use the model endpoint

The model ID can now be used by AlloyDB AI functions, for example `embedding()`. Here the query runs through `AlloyDBReader` and returns the number of dimensions of the embedding.

In [ ]:
from llama_index_alloydb_pg import AlloyDBReader

reader = await AlloyDBReader.create(
    engine,
    query=f"SELECT array_length(embedding('{MODEL_ID}', 'AlloyDB is a PostgreSQL-compatible database.'), 1) AS dimensions",
)
documents = await reader.aload_data()
print(documents[0].text)

## Clean up

Drop the model endpoint when you no longer need it.

In [ ]:
await manager.adrop_model(MODEL_ID)